# Graph Algorithms

Notes and runnable examples on the classic graph algorithms — shortest paths, spanning trees, and the union-find structure that powers them. This is where the data-structure notebooks pay off: every algorithm here is a `dict` graph plus a `heapq`, a `deque`, or a `set`, wired together. It builds directly on the **graphs** notebook (representations, BFS/DFS, topological sort).

**Contents**

1. **The landscape**
2. **Dijkstra** — weighted shortest paths
3. **Bellman-Ford** — negative weights & cycle detection
4. **A\*** — adding a heuristic
5. **Floyd–Warshall** — all-pairs shortest paths
6. **Union-Find** — the disjoint-set structure
7. **Minimum spanning tree** — Kruskal & Prim
8. **When to use what**


## 1. The landscape

The graphs notebook gave us the structure (adjacency lists), the traversals (BFS/DFS), and one algorithm per traversal: **BFS = unweighted shortest path**, and **topological sort** on a DAG. This notebook adds the weighted and optimization algorithms:

| Problem | Algorithm | Built from |
|---|---|---|
| Shortest path, non-negative weights | **Dijkstra** | a `heapq` priority queue |
| Shortest path, negative weights allowed | **Bellman-Ford** | an edge list + repeated relaxation |
| Shortest path to a goal, with a hint | **A\*** | `heapq` + a heuristic |
| Connectivity / "same group?" | **Union-Find** | arrays + path compression |
| Minimum spanning tree | **Kruskal / Prim** | union-find / `heapq` |

The throughline: each is a small twist on *explore the frontier in the right order* — and the right order is whatever a heap or a relaxation loop hands you.

## 2. Dijkstra — weighted shortest paths

**Dijkstra's algorithm** finds shortest paths from a source when all edge weights are **non-negative**. It's BFS with a priority queue: always expand the **closest** not-yet-finalized vertex next (via `heapq`), relaxing its edges. Once a vertex is popped with its smallest distance, that distance is final — which is exactly why negative weights break it (a later negative edge could undercut a "finalized" distance).

We use **lazy deletion**: instead of a decrease-key, we push a fresh `(distance, vertex)` entry and skip any **stale** pop whose recorded distance is already beaten. Tracking each vertex's predecessor lets us rebuild the actual path:

In [1]:
import heapq

def dijkstra(graph, start):
    dist = {start: 0}
    prev = {}
    pq = [(0, start)]                       # (distance_so_far, vertex)
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist.get(u, float("inf")):
            continue                        # stale entry - we already found a shorter way
        for v, w in graph[u]:
            nd = d + w
            if nd < dist.get(v, float("inf")):
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))
    return dist, prev

def path_to(prev, start, target):
    path = [target]
    while path[-1] != start:
        path.append(prev[path[-1]])
    return list(reversed(path))

graph = {
    "A": [("B", 1), ("C", 4)],
    "B": [("C", 2), ("D", 5)],
    "C": [("D", 1)],
    "D": [],
}
dist, prev = dijkstra(graph, "A")
print("distances from A  :", dist)
print("shortest path A->D:", path_to(prev, "A", "D"), "(A->B->C->D = 4 beats A->C->D = 5)")


distances from A  : {'A': 0, 'B': 1, 'C': 3, 'D': 4}
shortest path A->D: ['A', 'B', 'C', 'D'] (A->B->C->D = 4 beats A->C->D = 5)


## 3. Bellman-Ford — negative weights & cycle detection

When edges can be **negative**, Dijkstra's greedy finalization is unsafe. **Bellman-Ford** trades speed for generality: it just **relaxes every edge, V−1 times**. After V−1 passes every shortest path (which uses at most V−1 edges) has settled. A bonus falls out — if *one more* pass can still relax some edge, the graph has a **negative cycle**, so no shortest path exists. Cost is O(V·E): slower than Dijkstra's O(E log V), but it handles what Dijkstra can't.

In [2]:
def bellman_ford(vertices, edges, start):
    dist = {v: float("inf") for v in vertices}
    dist[start] = 0
    for _ in range(len(vertices) - 1):          # V-1 relaxation passes
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
    for u, v, w in edges:                        # one more pass: still improving?
        if dist[u] + w < dist[v]:
            return None                          # -> a negative cycle is reachable
    return dist

V = ["A", "B", "C", "D"]
E = [("A", "B", 4), ("A", "C", 5), ("B", "C", -3), ("C", "D", 2)]
print("with a negative edge B->C(-3):", bellman_ford(V, E, "A"))
print("  (C settles to 1 via A->B->C, not the direct A->C = 5)")

cyclic = [("A", "B", 1), ("B", "C", -2), ("C", "B", -2)]   # B<->C totals -4
print("negative cycle detected:", bellman_ford(["A", "B", "C"], cyclic, "A") is None)


with a negative edge B->C(-3): {'A': 0, 'B': 4, 'C': 1, 'D': 3}
  (C settles to 1 via A->B->C, not the direct A->C = 5)
negative cycle detected: True


## 4. A\* — adding a heuristic

When you're searching for **one specific goal** (not all shortest paths), you can beat Dijkstra by *guiding* the search toward the target. **A\*** orders the frontier by `f(n) = g(n) + h(n)` — distance-so-far plus a **heuristic** estimate of the distance remaining. With `h = 0` it *is* Dijkstra; with an **admissible** heuristic (one that never overestimates, like Manhattan distance on a grid) it skips vertices that obviously lead away from the goal.

Searching an open grid toward a goal off to the side, A\* expands far fewer cells than Dijkstra's blind expanding diamond:

In [3]:
import heapq

def grid_search(rows, cols, start, goal, heuristic, walls=frozenset()):
    def neighbors(r, c):
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in walls:
                yield (nr, nc)
    pq = [(heuristic(start, goal), 0, start)]   # (f = g + h, g, cell)
    dist = {start: 0}
    expanded = 0
    while pq:
        f, g, u = heapq.heappop(pq)
        if g > dist.get(u, float("inf")):
            continue
        expanded += 1
        if u == goal:
            return g, expanded
        for v in neighbors(*u):
            ng = g + 1
            if ng < dist.get(v, float("inf")):
                dist[v] = ng
                heapq.heappush(pq, (ng + heuristic(v, goal), ng, v))
    return None, expanded

manhattan = lambda a, b: abs(a[0] - b[0]) + abs(a[1] - b[1])
zero      = lambda a, b: 0                       # h = 0  ->  plain Dijkstra

start, goal = (7, 0), (7, 14)
d_dist, d_exp = grid_search(15, 15, start, goal, zero)
a_dist, a_exp = grid_search(15, 15, start, goal, manhattan)
print(f"Dijkstra (h=0)   : path {d_dist}, expanded {d_exp} cells")
print(f"A* (Manhattan h) : path {a_dist}, expanded {a_exp} cells  ({d_exp // a_exp}x fewer)")


Dijkstra (h=0)   : path 14, expanded 162 cells
A* (Manhattan h) : path 14, expanded 15 cells  (10x fewer)


Both find the **same** optimal distance — an admissible heuristic never sacrifices optimality, it only prunes wasted exploration. (A quirk worth knowing: on a *square* grid going corner-to-corner, every cell lies on some shortest path, so the heuristic can't prune and A\* degenerates to Dijkstra. The off-axis goal above is where it shines. Push `h` *above* the true cost and search gets faster still but may return non-optimal paths — that's the speed/optimality knob.)

## 5. Floyd–Warshall — all-pairs shortest paths

Dijkstra and Bellman-Ford each solve shortest paths from **one** source. When you need the distance between **every** pair of vertices, you could run them V times — but on a dense graph Floyd–Warshall does it in one tight triple loop, and it tolerates negative edges (as long as there's no negative cycle). It's pure DP: let `dist[i][j]` be the shortest path using only intermediate vertices from `{0..k}`; grow `k` one vertex at a time, asking "does routing through `k` beat what I already had?". A negative `dist[i][i]` afterwards exposes a negative cycle.

In [4]:
INF = float("inf")

def floyd_warshall(n, edges):
    dist = [[INF] * n for _ in range(n)]
    for i in range(n):
        dist[i][i] = 0
    for u, v, w in edges:                       # directed; add (v, u, w) for undirected
        dist[u][v] = min(dist[u][v], w)
    for k in range(n):                          # allow vertex k as an intermediate
        dk = dist[k]
        for i in range(n):
            dik = dist[i][k]
            if dik == INF:                      # i can't reach k -> k helps nothing
                continue
            di = dist[i]
            for j in range(n):
                if dik + dk[j] < di[j]:
                    di[j] = dik + dk[j]
    return dist

edges = [(0, 1, 3), (1, 2, 1), (0, 2, 7), (2, 3, 2), (3, 1, -3)]
dist = floyd_warshall(4, edges)
print("all-pairs distance matrix:")
for i, row in enumerate(dist):
    print(f"  from {i}:", ["%2d" % d if d < INF else " ." for d in row])
print("negative cycle?", any(dist[i][i] < 0 for i in range(4)))

# cross-check against Dijkstra-from-every-source on a random non-negative graph
import heapq, random
def dijkstra(n, adj, s):
    d = [INF] * n; d[s] = 0; pq = [(0, s)]
    while pq:
        du, u = heapq.heappop(pq)
        if du > d[u]:
            continue
        for v, w in adj[u]:
            if du + w < d[v]:
                d[v] = du + w; heapq.heappush(pq, (d[v], v))
    return d

random.seed(0)
n = 8; E = []; adj = [[] for _ in range(n)]
for u in range(n):
    for v in range(n):
        if u != v and random.random() < 0.4:
            w = random.randint(1, 9); E.append((u, v, w)); adj[u].append((v, w))
fw = floyd_warshall(n, E)
assert all(fw[s] == dijkstra(n, adj, s) for s in range(n))
print("Floyd-Warshall matches Dijkstra-from-every-source: OK")


all-pairs distance matrix:
  from 0: [' 0', ' 3', ' 4', ' 6']
  from 1: [' .', ' 0', ' 1', ' 3']
  from 2: [' .', '-1', ' 0', ' 2']
  from 3: [' .', '-3', '-2', ' 0']
negative cycle? False
Floyd-Warshall matches Dijkstra-from-every-source: OK


## 6. Union-Find — the disjoint-set structure

**Union-Find** (a.k.a. *disjoint-set union*, DSU) answers "are these two elements in the same group?" and "merge these groups" in **almost O(1)**. Each element points at a parent; the root identifies the set. Two optimizations make it nearly constant-time:

- **Path compression** — `find` flattens the tree, pointing nodes straight at the root as it walks up.
- **Union by rank** — always hang the shorter tree under the taller one.

Together they give an amortized cost of **O(α(n))** — the inverse Ackermann function, effectively a tiny constant. It's the backbone of Kruskal's MST and of any "connected components / equivalence classes" problem.

In [5]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))         # each element starts as its own root
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # path compression (halving)
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                     # already in the same set
        if self.rank[ra] < self.rank[rb]:    # attach the smaller tree under the larger
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
        return True

uf = UnionFind(5)
uf.union(0, 1)
uf.union(1, 2)
print("0 and 2 connected:", uf.find(0) == uf.find(2))     # True  (0-1-2)
print("0 and 3 connected:", uf.find(0) == uf.find(3))     # False


0 and 2 connected: True
0 and 3 connected: False


## 7. Minimum spanning tree — Kruskal & Prim

A **minimum spanning tree (MST)** connects every vertex of a weighted, undirected graph with the **least total edge weight** and no cycles. Two greedy algorithms find it, as mirror images:

- **Kruskal** — sort all edges cheapest-first; add an edge if it joins two *different* components (a **union-find** check), skip it if it would close a cycle.
- **Prim** — grow one tree from a start vertex, repeatedly taking the cheapest edge leaving it (a **`heapq`** of frontier edges).

Different strategies, identical minimum weight:

In [6]:
import heapq

def kruskal(n, edges):                       # edges: (weight, u, v)
    uf = UnionFind(n)
    total, tree = 0, []
    for w, u, v in sorted(edges):            # cheapest first
        if uf.union(u, v):                   # joins two components -> keep it
            total += w
            tree.append((u, v, w))
    return total, tree

def prim(n, adj, start=0):                   # adj[u] = [(neighbor, weight), ...]
    seen = [False] * n
    pq = [(0, start)]
    total = 0
    while pq:
        w, u = heapq.heappop(pq)
        if seen[u]:
            continue
        seen[u] = True
        total += w
        for v, wt in adj[u]:
            if not seen[v]:
                heapq.heappush(pq, (wt, v))
    return total

edges = [(2, 0, 1), (3, 1, 2), (5, 1, 4), (6, 0, 3), (7, 2, 4), (8, 1, 3), (9, 3, 4)]
adj = [[] for _ in range(5)]
for w, u, v in edges:
    adj[u].append((v, w)); adj[v].append((u, w))

total, tree = kruskal(5, edges)
print("Kruskal MST weight:", total, "| edges:", tree)
print("Prim    MST weight:", prim(5, adj))


Kruskal MST weight: 16 | edges: [(0, 1, 2), (1, 2, 3), (1, 4, 5), (0, 3, 6)]
Prim    MST weight: 16


## 8. When to use what

| Problem | Use | Time |
|---|---|:---:|
| Unweighted shortest path | BFS (graphs notebook) | O(V + E) |
| Shortest path, non-negative weights | Dijkstra | O((V + E) log V) |
| Shortest path, negative weights | Bellman-Ford | O(V · E) |
| Shortest path to one goal + heuristic | A\* | O((V + E) log V), fewer nodes in practice |
| All-pairs shortest paths | Floyd-Warshall | O(V³) |
| "Same component?" / incremental connectivity | Union-Find | ~O(1) amortized |
| Minimum spanning tree (sparse / edge list) | Kruskal | O(E log E) |
| Minimum spanning tree (dense / adjacency) | Prim | O(E log V) |
| Order a DAG by dependencies | topological sort (graphs notebook) | O(V + E) |

These are the load-bearing graph algorithms; most others (max-flow, strongly-connected components, bridges — see the **advanced graph algorithms** notebook) are refinements of the same *explore the frontier in the right order* idea — built, as always, on a `dict`, a `heapq`, and a `set`.